In [1]:
from  langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv

# This function will load all the variables from the .env file and will 
# make them available in the os.environ dictionary (env variables)
load_dotenv() 

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from  langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

Bro API KEY Variable exists


# Chain with Parallel Chains

In [2]:
# Task 1 Prompt

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "you are a movie summarizer"),
    ("human", "please summarize the movie in brief: {input}")
])

In [3]:
# Task 2 LLM
llm_openai = ChatOpenAI(model="gpt-5-mini",temperature=0)

In [ ]:
# Task 3 Str Parser -- removes the unncecessary content from the out and returns only string
str_parser = StrOutputParser()

In [7]:
# Task 4 Custom Runnable
from langchain_core.runnables import RunnableLambda

# we cannot pass or run a python function directly in the chain, 
# we need to convert/wrap it in a RunnableLambda

def dict_maker(text: str) -> dict:
    return {"text" : text}

dict_maker_runnable = RunnableLambda(dict_maker)

# Parallel Chain 1

In [ ]:
# task 1 prompt

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a linkedin post generator"),
    ("human", "create a post for the following text for linkedin: {text}")
])


#task 2 LLM
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)


#task 3 Str Parser
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser 

# Parallel Chain 2

In [13]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [11]:
def insta_chain(text:dict):

    text = text["text"]

    # task 1 prompt
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a linkedin post generator"),
    ("human", "create a post for the following text for Instagram: {text}")])


    #task 2 LLM
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)


    #task 3 Str Parser
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser
    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)





# Final Orchestration

In [ ]:
final_chain = (
    prompt_template | 
    llm_openai |
    str_parser |
    dict_maker_runnable |
    RunnableParallel(branches = {"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)


In [15]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'If you’re looking for a film that combines raw ambition, cinematic craft, and a stark look at power, KGF: Chapter 1 is worth a watch.\n\nQuick summary:\nRocky — a poor, driven young man in Bombay — promises his dying mother he’ll become powerful. His pursuit of wealth and status takes him to the brutal Kolar Gold Fields, where miners suffer under a ruthless crime lord and corrupt officials. Rocky rises through violence and cunning to become a feared force who aims to take control of the mines and change the status quo. Stylized, high-energy, and set against a gritty 1970s–80s backdrop, the film stars Yash and is directed by Prashanth Neel.\n\nBusiness and leadership takeaways:\n- Ambition needs purpose: Rocky’s drive is rooted in a promise — ambition guided by a clear “why” can be transformative.  \n- Power and responsibility: The film raises questions about how power is gained and used — a reminder that leadership without ethics can easily become oppression.

# Chain as a Runnable

In [17]:
# task 1 Beautify Function

def beautify(final_response: dict)-> dict:
    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


#task 2 final chain

#final_chain

# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")


{'linkedin': 'Just watched Pushpa: The Rise — a gritty Telugu action‑drama from director Sukumar that’s as much about survival and ambition as it is about crime.\n\nAt the center is Pushpa Raj (Allu Arjun), a downtrodden labourer who claws his way up the illicit red sandalwood‑smuggling trade in the Seshachalam forests. Using cunning, violence and sheer ambition, he builds a criminal network, clashes with rivals and law enforcement, and tries to protect his family while pursuing his love interest Srivalli (Rashmika Mandanna). The film’s pulse comes from a raw central performance, pounding music and high‑voltage action — all underscoring themes of class struggle, ambition and what people will do to survive.\n\nTakeaways for leaders and creatives: strong storytelling can humanize morally ambiguous characters; resilience and resourcefulness matter when systems are stacked against you; and craft—performance, score, direction—can elevate a familiar arc into something visceral.\n\nHave you s